<a href="https://colab.research.google.com/github/crialejo24/DOWNSCALING/blob/main/SwinIR_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load libraries and Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
#!pip install timm
#!pip install einops

from huggingface_hub import HfApi
import os, subprocess

!pip install -q huggingface_hub

Mounted at /content/drive


# Section 1: New training

## Load official repository

In [ ]:
#REPOSITORIO ORIGNAL
!git clone https://github.com/cszn/KAIR.git

%cd KAIR
!pip install -r requirement.txt

## Select training type

### You can find the two fine-tuning training configurations shown in the figure in the "Downscaling" GitHub repository by author crialejo24.

![SwinIR](https://imgur.com/4r1eLvr.jpg)

In [ ]:
#training='training1'
training='training2'

## Replace modified files

In [ ]:
import os
import urllib.request
import shutil

def select_training_type(training):

    # Verificar selección
    if training == "training1":
        carpeta = "modificables_swinir_train_1"
    elif training == "training2":
        carpeta = "modificables_swinir_train_2"
    else:
        raise ValueError("La opción debe ser 'training1' o 'training2'.")

    # Archivos que se van a descargar
    archivos = {
        "model_plain.py": f"https://raw.githubusercontent.com/crialejo24/DOWNSCALING/main/{carpeta}/model_plain.py",
        "utils_image.py": f"https://raw.githubusercontent.com/crialejo24/DOWNSCALING/main/{carpeta}/utils_image.py"
    }

    # Destinos dentro de KAIR
    destinos = {
        "model_plain.py": "/content/KAIR/models/model_plain.py",
        "utils_image.py": "/content/KAIR/utils/utils_image.py"
    }

    # Descargar archivos
    for archivo, url in archivos.items():
        destino = destinos[archivo]

        print(f"Descargando {archivo} desde {carpeta}...")

        urllib.request.urlretrieve(url, destino)

        print(f"✓ {archivo} actualizado")

    print(f"\nConfiguración seleccionada: {training}")

In [ ]:
select_training_type(training)

Descargando model_plain.py desde modificables_swinir_train_2...
✓ model_plain.py actualizado
Descargando utils_image.py desde modificables_swinir_train_2...
✓ utils_image.py actualizado

Configuración seleccionada: training2


If you need to modify the trainable layers, load either of the two configurations and edit them in the following file: /content/KAIR/models/model_plain.py

## Load pretrained model

In [ ]:
!wget https://github.com/JingyunLiang/SwinIR/releases/download/v0.0/001_classicalSR_DF2K_s64w8_SwinIR-M_x3.pth -P model_zoo/
!ls model_zoo

--2026-09-11 16:49:32--  https://github.com/JingyunLiang/SwinIR/releases/download/v0.0/001_classicalSR_DF2K_s64w8_SwinIR-M_x3.pth
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/396770997/c3116ff3-6b2f-47f3-96d7-5c9bb8db3054?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-09-11T17%3A46%3A43Z&rscd=attachment%3B+filename%3D001_classicalSR_DF2K_s64w8_SwinIR-M_x3.pth&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-09-11T16%3A46%3A42Z&ske=2026-09-11T17%3A46%3A43Z&sks=b&skv=2018-11-09&sig=RXGshIAUZ%2BO2%2FwAXBFNnxlAEpns4DRmEUdUr4%2BGLPzg%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4OTE0NzE3MiwibmJmIjox

## Upload dataset images

In [ ]:
!mkdir -p trainsets/trainH
!mkdir -p trainsets/trainL
!mkdir -p testsets/testH
!mkdir -p testsets/testL

api = HfApi()

for hf, dst in [
    ("trainset/HR_192_mod", "/content/KAIR/trainsets/trainH"),
    ("trainset/LR_64_modx3", "/content/KAIR/trainsets/trainL"),
    ("testset/HR_192_mod", "/content/KAIR/testsets/testH"),
    ("testset/LR_64_modx3", "/content/KAIR/testsets/testL")
]:
    os.makedirs(dst, exist_ok=True)
    for f in api.list_repo_tree("Crialejo924/DOWNSCALING", path_in_repo=hf, repo_type="dataset", recursive=True):
        if hasattr(f, "path") and f.path.endswith(".tif"):
            subprocess.run(["wget", "-q", "--show-progress",
                f"https://huggingface.co/datasets/Crialejo924/DOWNSCALING/resolve/main/{f.path}",
                "-P", dst])

## Training Configuration

It is important to assign a path external to Colab to the "root" item in the .json file in order to save training progress and avoid losing it if the Colab session closes or restarts. By default, the user's Google Drive will be used.

You must also specify the path to the pre-trained model in the "pretrained_netG" item.

In [ ]:
progress_folder = f'/content/drive/MyDrive/Repository_SwinIR_{training}'

os.makedirs(progress_folder, exist_ok=True)

In [ ]:
import json
import os

os.makedirs("options/swinir", exist_ok=True)

config = {
  "task": "swinir_sr_x3_finetune"     #classical image sr for x2/x3/x4/x8. root/task/images-models-options
  , "model": "plain" # "plain" | "plain2" if two inputs
  , "gpu_ids": [0]
  , "dist": False

  , "scale": 3       # 2 | 3 | 4 | 8
  , "n_channels": 3  # broadcast to "datasets", 1 for grayscale, 3 for color

  , "path": {
    "root": f'{progress_folder}/superresolution' # Ruta donde se guarda el avance del entrenamiento
    , "pretrained_netG": "model_zoo/001_classicalSR_DF2K_s64w8_SwinIR-M_x3.pth"      # path of pretrained model. We fine-tune X3/X4/X8 models from X2 model, so that `G_optimizer_lr` and `G_scheduler_milestones` can be halved to save time.
    , "pretrained_netE": None      # path of pretrained model
  }

  , "datasets": {
    "train": {
      "name": "train_dataset"           #// just name
      , "dataset_type": "sr"         #// "dncnn" | "dnpatch" | "fdncnn" | "ffdnet" | "sr" | "srmd" | "dpsr" | "plain" | "plainpatch" | "jpeg"
      , "dataroot_H": "trainsets/trainH" #// path of H training dataset. DIV2K (800 training images)
      , "dataroot_L": "trainsets/trainL"  #            // path of L training dataset

      , "H_size": 192                   #// 96/144|192/384 | 128/192/256/512. LR patch size is set to 48 or 64 when compared with RCAN or RRDB.

      , "dataloader_shuffle": True
      , "dataloader_num_workers": 4
      , "dataloader_batch_size": 8      #// batch size 1 | 16 | 32 | 48 | 64 | 128. Total batch size =4x8=32 in SwinIR
    }
    , "test": {
      "name": "test_dataset"            #// just name
      , "dataset_type": "sr"         #// "dncnn" | "dnpatch" | "fdncnn" | "ffdnet" | "sr" | "srmd" | "dpsr" | "plain" | "plainpatch" | "jpeg"
      , "dataroot_H": "testsets/testH"  #// path of H testing dataset
      , "dataroot_L": "testsets/testL"              #// path of L testing dataset

    }
  }

  , "netG": {
    "net_type": "swinir"
    , "upscale": 3                      #// 2 | 3  | 4 | 8
    , "in_chans": 3
    , "img_size": 64                    #// For fair comparison, LR patch size is set to 48 or 64 when compared with RCAN or RRDB.
    , "window_size": 8
    , "img_range": 255
    , "depths": [6, 6, 6, 6, 6, 6]
    , "embed_dim": 180
    , "num_heads": [6, 6, 6, 6, 6, 6]
    , "mlp_ratio": 2
    , "upsampler": "pixelshuffle"        #// "pixelshuffle" | "pixelshuffledirect" | "nearest+conv" | null
    , "resi_connection": "1conv"        #// "1conv" | "3conv"

    , "init_type": "default"
  }

  , "train": {
    "G_lossfn_type": "l1"               #// "l1" preferred | "l2sum" | "l2" | "ssim" | "charbonnier"
    , "G_lossfn_weight": 1.0            #// default

    , "E_decay": 0.999                  #// Exponential Moving Average for netG: set 0 to disable; default setting 0.999

    , "G_optimizer_type": "adam"        #// fixed, adam is enough
    , "G_optimizer_lr": 2e-4            #// learning rate
    , "G_optimizer_wd": 0               #// weight decay, default 0
    , "G_optimizer_clipgrad": None      #// unused
    , "G_optimizer_reuse": True         #//

    , "G_scheduler_type": "MultiStepLR" #// "MultiStepLR" is enough
    , "G_scheduler_milestones": [250000, 400000, 450000, 475000, 500000]
    , "G_scheduler_gamma": 0.5

    , "G_regularizer_orthstep": None    #// unused
    , "G_regularizer_clipstep": None    #// unused

    , "G_param_strict": True
    , "E_param_strict": True

    , "checkpoint_test": 5000           #// for testing
    , "checkpoint_save": 5000           #// for saving model
    , "checkpoint_print": 200           #// for print
  }
}

with open("options/swinir/swinir_sr_x3_finetune.json", "w") as f:
    json.dump(config, f, indent=2)

print("JSON creado en options/swinir/")

JSON creado en options/swinir/


## Save the repository containing the dataset and training configuration to Google Drive.

In [ ]:
!cp -r /content/KAIR/* "{progress_folder}"


## Execution of the training

In [ ]:
!python main_train_psnr.py --opt options/swinir/swinir_sr_x3_finetune.json

export CUDA_VISIBLE_DEVICES=0
number of GPUs is: 1
LogHandlers setup!
26-09-11 17:41:52.539 :   task: swinir_sr_x3_finetune
  model: plain
  gpu_ids: [0]
  dist: False
  scale: 3
  n_channels: 3
  path:[
    root: /content/drive/MyDrive/Repository_SwinIR_training2/superresolution
    pretrained_netG: None
    pretrained_netE: None
    task: /content/drive/MyDrive/Repository_SwinIR_training2/superresolution/swinir_sr_x3_finetune
    log: /content/drive/MyDrive/Repository_SwinIR_training2/superresolution/swinir_sr_x3_finetune
    options: /content/drive/MyDrive/Repository_SwinIR_training2/superresolution/swinir_sr_x3_finetune/options
    models: /content/drive/MyDrive/Repository_SwinIR_training2/superresolution/swinir_sr_x3_finetune/models
    images: /content/drive/MyDrive/Repository_SwinIR_training2/superresolution/swinir_sr_x3_finetune/images
    pretrained_optimizerG: None
  ]
  datasets:[
    train:[
      name: train_dataset
      dataset_type: sr
      dataroot_H: trainsets/trainH

# Section 2: Resume training from a repository saved in Drive.

The following section allows you to resume a training session that was started previously and saved to Drive. Verify that the Google Drive folder name is correct.

In [ ]:
#training='training1'
training='training2'

## Load repository

In [ ]:
folder=f'/content/drive/MyDrive/Repository_SwinIR_{training}'

#Training repository
!cp -r "{folder}" /content/KAIR

%cd KAIR
!pip install -r requirement.txt

/content/KAIR
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.6/77.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.8/346.8 kB 34.7 MB/s eta 0:00:00


## Resume training

In [ ]:
!python main_train_psnr.py --opt options/swinir/swinir_sr_x3_finetune.json

export CUDA_VISIBLE_DEVICES=0
number of GPUs is: 1
LogHandlers setup!
26-09-11 18:15:02.109 :   task: swinir_sr_x3_finetune
  model: plain
  gpu_ids: [0]
  dist: False
  scale: 3
  n_channels: 3
  path:[
    root: /content/drive/MyDrive/Repository_SwinIR_training2/superresolution
    pretrained_netG: None
    pretrained_netE: None
    task: /content/drive/MyDrive/Repository_SwinIR_training2/superresolution/swinir_sr_x3_finetune
    log: /content/drive/MyDrive/Repository_SwinIR_training2/superresolution/swinir_sr_x3_finetune
    options: /content/drive/MyDrive/Repository_SwinIR_training2/superresolution/swinir_sr_x3_finetune/options
    models: /content/drive/MyDrive/Repository_SwinIR_training2/superresolution/swinir_sr_x3_finetune/models
    images: /content/drive/MyDrive/Repository_SwinIR_training2/superresolution/swinir_sr_x3_finetune/images
    pretrained_optimizerG: None
  ]
  datasets:[
    train:[
      name: train_dataset
      dataset_type: sr
      dataroot_H: trainsets/trainH